# Assignment 2 DSC 102 FA23

## Introduction

In this assignment we will conduct data engineering for the Amazon dataset. It is divided into 2 parts. The extracted features in Part 1 will be used for the Part 2 of assignment, where you train a model (or models) to predict user ratings for a product.

We will be using Apache Spark for this assignment. The default Spark API will be DataFrame, as it is now the recommended choice over the RDD API. That being said, please feel free to switch back to the RDD API if you see it as a better fit for the task. We provide you an option to request RDD format to start with. Also you can switch between DataFrame and RDD in your solution. 

Another newer API is Koalas, which is also avaliable. However, it has constraints and is not applicable to most tasks. Refer to the PA statement for detail.

### Set the following parameters

In [1]:
PID = 'a17787465' # your pid, for instance: 'a43223333'
INPUT_FORMAT = 'dataframe' # choose a format of your input data, valid options: 'dataframe', 'rdd', 'koalas'

In [2]:
# Boiler plates, do NOT modify
%load_ext autoreload
%autoreload 2
import os
import getpass
from pyspark.sql import SparkSession
from utilities import SEED
from utilities import PA2Test
from utilities import PA2Data
from utilities import data_cat
from pa2_main import PA2Executor
import time
if INPUT_FORMAT == 'dataframe':
    import pyspark.ml as M
    import pyspark.sql.functions as F
    import pyspark.sql.types as T
if INPUT_FORMAT == 'koalas':
    import databricks.koalas as ks
elif INPUT_FORMAT == 'rdd':
    import pyspark.mllib as M
    from pyspark.mllib.feature import Word2Vec
    from pyspark.mllib.linalg import Vectors
    from pyspark.mllib.linalg.distributed import RowMatrix

os.environ['PYSPARK_SUBMIT_ARGS'] = '--py-files utilities.py,assignment2.py \
--deploy-mode client \
pyspark-shell'

class args:
    review_filename = data_cat.review_filename
    product_filename = data_cat.product_filename
    product_processed_filename = data_cat.product_processed_filename
    ml_features_train_filename = data_cat.ml_features_train_filename
    ml_features_test_filename = data_cat.ml_features_test_filename
    output_root = '/home/{}/{}-pa2/test_results'.format(getpass.getuser(), PID)
    test_results_root = data_cat.test_results_root
    pid = PID

pa2 = PA2Executor(args, input_format=INPUT_FORMAT)
data_io = pa2.data_io
data_dict = pa2.data_dict
begin = time.time()


25/05/16 02:16:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/opt/bitnami/spark/python/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Loading datasets ...Done



[Stage 17:>                                                         (0 + 1) / 1]



In [3]:
# Import your own dependencies



#-----------------------------

# Part 1: Feature Engineering

In [4]:
# Bring the part_1 datasets to memory and de-cache part_2 datasets. 
# Execute this once before you start working on this Part
data_dict, _ = data_io.cache_switch(data_dict, 'part_1')

# Task0: warm up 
This task is provided for you to get familiar with Spark API. We will use the dataframe API to demonstrate. Solution is given to you and this task won't be graded.

Refer to https://spark.apache.org/docs/latest/api/python/pyspark.sql.html for API guide.

The task is to implement the function below. Given the ```product_data``` table:
1. Take and print five rows.

1. Select only the ```asin``` column, then print five rows of it.

1. Select the row where ```asin = B00I8KEOTM``` and print it.

1. Count the total number of rows.

1. Calculate the mean ```price```.

1. You need to conduct the above operations, then extract some statistics out of the generated columns. You need to put the statistics in a python dictionary named ```res```. The description and schema of it are as follows:
    ```
    res
     | -- count_total: int -- count of total rows of the entire table after your operations
     | -- mean_price: float -- mean value of column price
    ```

In [5]:
def task_0(data_io, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    asin_column = 'asin'
    overall_column = 'overall'
    # Outputs:
    mean_rating_column = 'meanRating'
    count_rating_column = 'countRating'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------

    product_data.show(5)
    product_data[['asin']].show(5)
    product_data.where(F.col('asin') == 'B00I8KEOTM').show()
    count_rows = product_data.count()
    mean_price = product_data.select(F.avg(F.col('price'))).head()[0]
    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    # Calculate the values programmatically. Do not change the keys and do not
    # hard-code values in the dict. Your submission will be evaluated with
    # different inputs.
    # Modify the values of the following dictionary accordingly.
    res = {'count_total': None, 'mean_price': None}
    
    # Modify res:

    res['count_total'] = count_rows
    res['mean_price'] = mean_price

    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    return res
    # -------------------------------------------------------------------------

In [6]:
if INPUT_FORMAT == 'dataframe':
    res = task_0(data_io, data_dict['product'])
    pa2.tests.test(res, 'task_0')

+----------+--------------------+--------------------+--------------------+-----+--------------------+
|      asin|           salesRank|          categories|               title|price|             related|
+----------+--------------------+--------------------+--------------------+-----+--------------------+
|B00I8HVV6E|{Home &amp; Kitch...|[[Home & Kitchen,...|Intelligent Desig...|27.99|{also_viewed -> [...|
|B00I8KEOTM|                null|[[Apps for Androi...|                null| null|{also_viewed -> [...|
|B00I8KCW4G|{Clothing -> 2233...|[[Clothing, Shoes...|eShakti Women's P...|41.95|{also_viewed -> [...|
|B00I8JKCQW|{Clothing -> 1405...|[[Clothing, Shoes...|Lady Slimming Mid...| null|{also_viewed -> [...|
|B00I8JKI8E|{Home &amp; Kitch...|[[Clothing, Shoes...|3 Tier Bangle Bra...|24.99|{also_viewed -> [...|
+----------+--------------------+--------------------+--------------------+-----+--------------------+
only showing top 5 rows

+----------+
|      asin|
+----------+
|B00I8HVV

tests for task_0 --------------------------------------------------------------
2/2 passed
-------------------------------------------------------------------------------


# Task1

In [7]:
# %load -s task_1 assignment2.py
def task_1(data_io, review_data, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    asin_column = 'asin'
    overall_column = 'overall'
    # Outputs:
    mean_rating_column = 'meanRating'
    count_rating_column = 'countRating'

    # ---------------------- Your implementation begins------------------------
    # Calculate mean and count of ratings for each product
    ratings_agg = review_data.groupBy(asin_column) \
        .agg(
            F.avg(overall_column).alias(mean_rating_column),
            F.count(overall_column).alias(count_rating_column)
        )
    
    # Join with product data to include products with no reviews
    result = product_data.join(ratings_agg, asin_column, 'left')

    res = {
        'count_total': result.count(),
        'mean_meanRating': result.select(F.avg(mean_rating_column)).head()[0],
        'variance_meanRating': result.select(F.variance(mean_rating_column)).head()[0],
        'numNulls_meanRating': result.filter(F.col(mean_rating_column).isNull()).count(),
        'mean_countRating': result.select(F.avg(count_rating_column)).head()[0],
        'variance_countRating': result.select(F.variance(count_rating_column)).head()[0],
        'numNulls_countRating': result.filter(F.col(count_rating_column).isNull()).count()
    }

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_1')
    return res
    # -------------------------------------------------------------------------

In [8]:
res = task_1(data_io, data_dict['review'], data_dict['product'])
pa2.tests.test(res, 'task_1')

tests for task_1 --------------------------------------------------------------
Test 1/7 : count_total ... Pass
Test 2/7 : mean_countRating ... Pass
Test 3/7 : mean_meanRating ... Pass
Test 4/7 : numNulls_countRating ... Pass
Test 5/7 : numNulls_meanRating ... Pass
Test 6/7 : variance_countRating ... Pass
Test 7/7 : variance_meanRating ... Pass
7/7 passed
-------------------------------------------------------------------------------


True


# Task 2

In [ ]:
# %load -s task_2 assignment2.py
def task_2(data_io, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    salesRank_column = 'salesRank'
    categories_column = 'categories'
    asin_column = 'asin'
    # Outputs:
    category_column = 'category'
    bestSalesCategory_column = 'bestSalesCategory'
    bestSalesRank_column = 'bestSalesRank'

    # ---------------------- Your implementation begins------------------------

    result = product_data.withColumn(
        category_column,
        F.when(
            (F.col(categories_column).isNull()) |
            (F.element_at(F.element_at(F.col(categories_column), 1), 1) == ""),
            F.lit(None)
        ).otherwise(F.element_at(F.element_at(F.col(categories_column), 1), 1))
    )
    
    # Flatten salesRank into two columns
    result = result.withColumn(
        bestSalesCategory_column,
        F.when(
            F.col(salesRank_column).isNull(),
            F.lit(None)
        ).otherwise(F.map_keys(F.col(salesRank_column))[0])
    ).withColumn(
        bestSalesRank_column,
        F.when(
            F.col(salesRank_column).isNull(),
            F.lit(None)
        ).otherwise(F.map_values(F.col(salesRank_column))[0])
    )

    res = {
        'count_total': result.count(),
        'mean_bestSalesRank': result.select(F.avg(bestSalesRank_column)).head()[0],
        'variance_bestSalesRank': result.select(F.variance(bestSalesRank_column)).head()[0],
        'numNulls_category': result.filter(F.col(category_column).isNull()).count(),
        'countDistinct_category': result.select(F.countDistinct(category_column)).head()[0],
        'numNulls_bestSalesCategory': result.filter(F.col(bestSalesCategory_column).isNull()).count(),
        'countDistinct_bestSalesCategory': result.select(F.countDistinct(bestSalesCategory_column)).head()[0]
    }

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_2')
    return res
    # -------------------------------------------------------------------------

In [ ]:
res = task_2(data_io, data_dict['product'])
pa2.tests.test(res, 'task_2')

# Task 3





In [ ]:
def task_3(data_io, product_data):
    start = time.time()
    # -----------------------------Column names--------------------------------
    # Inputs:
    asin_column = 'asin'
    price_column = 'price'
    attribute = 'also_viewed'
    related_column = 'related'
    # Outputs:
    meanPriceAlsoViewed_column = 'meanPriceAlsoViewed'
    countAlsoViewed_column = 'countAlsoViewed'
    
    # -------------------------------------------------------------------------
    # Explode also_viewed into individual product IDs
    exploded = product_data.select(
        '*',
        F.explode_outer(F.col(related_column)[attribute]).alias('viewed_id')
    )

    # Join with product data to get prices
    joined = exploded.join(
        product_data.select(
            F.col(asin_column).alias('joined_asin'),
            F.col(price_column).alias('joined_price')
        ),
        exploded['viewed_id'] == F.col('joined_asin'),
        how='left'
    )

    # Aggregate mean prices per row
    aggregated = joined.groupBy(asin_column).agg(
        F.mean('joined_price').alias(meanPriceAlsoViewed_column),
        F.when(
            F.count('viewed_id') == 0,
            F.lit(None)
        ).otherwise(F.count('viewed_id')).alias(countAlsoViewed_column)
    )

    # Join back to the original data to ensure we preserve all rows
    final = product_data.join(aggregated, on=asin_column, how='left')

    # Statistics
    count_total = final.count()
    mean_meanPriceAlsoViewed = final.select(F.avg(meanPriceAlsoViewed_column)).first()[0]
    variance_meanPriceAlsoViewed = final.select(F.variance(meanPriceAlsoViewed_column)).first()[0]
    numNulls_meanPriceAlsoViewed = final.filter(F.col(meanPriceAlsoViewed_column).isNull()).count()
    mean_countAlsoViewed = final.select(F.avg(countAlsoViewed_column)).first()[0]
    variance_countAlsoViewed = final.select(F.variance(countAlsoViewed_column)).first()[0]
    numNulls_countAlsoViewed = final.filter(F.col(countAlsoViewed_column).isNull()).count()


    # ---------------------- Compute statistics --------------------------
    res = {
        'count_total': count_total,
        'mean_meanPriceAlsoViewed': mean_meanPriceAlsoViewed,
        'variance_meanPriceAlsoViewed': variance_meanPriceAlsoViewed,
        'numNulls_meanPriceAlsoViewed': numNulls_meanPriceAlsoViewed,
        'mean_countAlsoViewed': mean_countAlsoViewed,
        'variance_countAlsoViewed': variance_countAlsoViewed,
        'numNulls_countAlsoViewed': numNulls_countAlsoViewed
    }

    data_io.save(res, 'task_3')
    print(time.time() - start)
    return res

In [ ]:
res = task_3(data_io, data_dict['product'])
pa2.tests.test(res, 'task_3')

# Task 4

In [ ]:
def task_4(data_io, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    price_column = 'price'
    title_column = 'title'
    # Outputs:
    meanImputedPrice_column = 'meanImputedPrice'
    medianImputedPrice_column = 'medianImputedPrice'
    unknownImputedTitle_column = 'unknownImputedTitle'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------
    # Cast price to float and calculate mean and median
    price_stats = product_data.select(
        F.avg(F.col(price_column).cast('float')).alias('mean_price'),
        F.expr('percentile(cast(price as float), 0.5)').alias('median_price')
    ).head()
    
    mean_price = price_stats['mean_price']
    median_price = price_stats['median_price']
    
    # Create result with imputed values
    result = product_data.withColumn(
        meanImputedPrice_column,
        F.when(
            F.col(price_column).isNull(),
            F.lit(mean_price)
        ).otherwise(F.col(price_column).cast('float'))
    ).withColumn(
        medianImputedPrice_column,
        F.when(
            F.col(price_column).isNull(),
            F.lit(median_price)
        ).otherwise(F.col(price_column).cast('float'))
    ).withColumn(
        unknownImputedTitle_column,
        F.when(
            (F.col(title_column).isNull()) | (F.col(title_column) == ''),
            F.lit('unknown')
        ).otherwise(F.col(title_column))
    )
    
    # Calculate statistics
    count_total = result.count()
    mean_meanImputedPrice = result.select(F.avg(meanImputedPrice_column)).head()[0]
    variance_meanImputedPrice = result.select(F.variance(meanImputedPrice_column)).head()[0]
    numNulls_meanImputedPrice = result.filter(F.col(meanImputedPrice_column).isNull()).count()
    mean_medianImputedPrice = result.select(F.avg(medianImputedPrice_column)).head()[0]
    variance_medianImputedPrice = result.select(F.variance(medianImputedPrice_column)).head()[0]
    numNulls_medianImputedPrice = result.filter(F.col(medianImputedPrice_column).isNull()).count()
    numUnknowns_unknownImputedTitle = result.filter(F.col(unknownImputedTitle_column) == 'unknown').count()

    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': count_total,
        'mean_meanImputedPrice': mean_meanImputedPrice,
        'variance_meanImputedPrice': variance_meanImputedPrice,
        'numNulls_meanImputedPrice': numNulls_meanImputedPrice,
        'mean_medianImputedPrice': mean_medianImputedPrice,
        'variance_medianImputedPrice': variance_medianImputedPrice,
        'numNulls_medianImputedPrice': numNulls_medianImputedPrice,
        'numUnknowns_unknownImputedTitle': numUnknowns_unknownImputedTitle
    }
    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_4')
    return res
    # -------------------------------------------------------------------------

In [ ]:
res = task_4(data_io, data_dict['product'])
pa2.tests.test(res, 'task_4')

# Task 5

In [ ]:
def task_5(data_io, product_processed_data, word_0, word_1, word_2):
    # -----------------------------Column names--------------------------------
    # Inputs:
    title_column = 'title'
    # Outputs:
    titleArray_column = 'titleArray'
    titleVector_column = 'titleVector'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------
    # Convert title to lowercase and split by whitespace
    result = product_processed_data.withColumn(
        titleArray_column,
        F.split(F.lower(F.col(title_column)), ' ')
    )
    
    # Train Word2Vec model
    word2Vec = M.feature.Word2Vec(
        minCount=100,
        vectorSize=16,
        seed=SEED,
        numPartitions=4
    )
    
    # Fit the model
    model = word2Vec.fit(result.select(titleArray_column))
    
    # Transform the data to get word vectors
    result = model.transform(result)
    
    # Calculate statistics
    count_total = result.count()
    size_vocabulary = model.getVectors().count()
    
    # Get synonyms for each word
    word_0_synonyms = model.findSynonymsArray(word_0, 10)
    word_1_synonyms = model.findSynonymsArray(word_1, 10)
    word_2_synonyms = model.findSynonymsArray(word_2, 10)

    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': count_total,
        'size_vocabulary': size_vocabulary,
        'word_0_synonyms': word_0_synonyms,
        'word_1_synonyms': word_1_synonyms,
        'word_2_synonyms': word_2_synonyms
    }
    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_5')
    return res
    # ------------------------------------------------------------------------

In [ ]:
res = task_5(data_io, data_dict['product_processed'], 'piano', 'rice', 'laptop')
pa2.tests.test(res, 'task_5')

# Task 6

In [ ]:
def task_6(data_io, product_processed_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    category_column = 'category'
    # Outputs:
    categoryIndex_column = 'categoryIndex'
    categoryOneHot_column = 'categoryOneHot'
    categoryPCA_column = 'categoryPCA'
    # -------------------------------------------------------------------------    

    # ---------------------- Your implementation begins------------------------
    # Convert category to index
    indexer = M.feature.StringIndexer(
        inputCol=category_column,
        outputCol=categoryIndex_column,
        handleInvalid='keep'
    )
    
    # One-hot encode the indexed categories
    encoder = M.feature.OneHotEncoder(
        inputCol=categoryIndex_column,
        outputCol=categoryOneHot_column,
        dropLast=False
    )
    
    # Create pipeline for indexing and encoding
    pipeline = M.Pipeline(stages=[indexer, encoder])
    model = pipeline.fit(product_processed_data)
    result = model.transform(product_processed_data)
    
    # Apply PCA
    pca = M.feature.PCA(
        k=15,
        inputCol=categoryOneHot_column,
        outputCol=categoryPCA_column
    )
    pca_model = pca.fit(result)
    result = pca_model.transform(result)
    
    # Calculate statistics
    count_total = result.count()
    
    # Calculate mean vectors using Summarizer
    summarizer = M.stat.Summarizer.metrics("mean")
    meanVector_categoryOneHot = result.select(
        summarizer.summary(F.col(categoryOneHot_column))
    ).head()[0].mean.toArray().tolist()
    
    meanVector_categoryPCA = result.select(
        summarizer.summary(F.col(categoryPCA_column))
    ).head()[0].mean.toArray().tolist()

    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': count_total,
        'meanVector_categoryOneHot': meanVector_categoryOneHot,
        'meanVector_categoryPCA': meanVector_categoryPCA
    }
    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_6')
    return res
    # -------------------------------------------------------------------------

In [ ]:
res = task_6(data_io, data_dict['product_processed'])
pa2.tests.test(res, 'task_6')

In [ ]:
print ("End to end time: {}".format(time.time()-begin))

# Part 2: Model Selection

In [ ]:
# Bring the part_2 datasets to memory and de-cache part_1 datasets.
# Execute this once before you start working on this Part
data_dict, _ = data_io.cache_switch(data_dict, 'part_2')

# Task 7

In [ ]:
def task_7(data_io, train_data, test_data):
    # ---------------------- Your implementation begins------------------------
    # Prepare features and label
    feature_cols = [col for col in train_data.columns if col != 'overall']
    
    # Create feature vector
    assembler = M.feature.VectorAssembler(
        inputCols=feature_cols,
        outputCol='features'
    )
    
    # Create Decision Tree model
    dt = M.regression.DecisionTreeRegressor(
        maxDepth=5,
        featuresCol='features',
        labelCol='overall'
    )
    
    # Create pipeline
    pipeline = M.Pipeline(stages=[assembler, dt])
    
    # Train model
    model = pipeline.fit(train_data)
    
    # Make predictions on test data
    predictions = model.transform(test_data)
    
    # Calculate RMSE
    evaluator = M.evaluation.RegressionEvaluator(
        labelCol='overall',
        predictionCol='prediction',
        metricName='rmse'
    )
    test_rmse = evaluator.evaluate(predictions)
    
    # -------------------------------------------------------------------------
    
    # ---------------------- Put results in res dict --------------------------
    res = {
        'test_rmse': test_rmse
    }
    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_7')
    return res
    # -------------------------------------------------------------------------

In [ ]:
res = task_7(data_io, data_dict['ml_features_train'], data_dict['ml_features_test'])
pa2.tests.test(res, 'task_7')

# Task 8

In [ ]:
def task_8(data_io, train_data, test_data):
    # ---------------------- Your implementation begins------------------------
    # Split training data into train and validation sets (75/25)
    train_data, valid_data = train_data.randomSplit([0.75, 0.25], seed=SEED)
    
    # Prepare features and label
    feature_cols = [col for col in train_data.columns if col != 'overall']
    
    # Create feature vector
    assembler = M.feature.VectorAssembler(
        inputCols=feature_cols,
        outputCol='features'
    )
    
    # Create evaluator
    evaluator = M.evaluation.RegressionEvaluator(
        labelCol='overall',
        predictionCol='prediction',
        metricName='rmse'
    )
    
    # Dictionary to store validation RMSEs
    valid_rmses = {}
    
    # Train models with different depths and calculate validation RMSEs
    for depth in [5, 7, 9, 12]:
        # Create Decision Tree model
        dt = M.regression.DecisionTreeRegressor(
            maxDepth=depth,
            featuresCol='features',
            labelCol='overall'
        )
        
        # Create pipeline
        pipeline = M.Pipeline(stages=[assembler, dt])
        
        # Train model
        model = pipeline.fit(train_data)
        
        # Make predictions on validation data
        valid_predictions = model.transform(valid_data)
        
        # Calculate validation RMSE
        valid_rmse = evaluator.evaluate(valid_predictions)
        valid_rmses[depth] = valid_rmse
        
        # If this is the best model so far, save it for test predictions
        if depth == 5 or valid_rmse < best_valid_rmse:
            best_model = model
            best_valid_rmse = valid_rmse
            best_depth = depth
    
    # Make predictions on test data using best model
    test_predictions = best_model.transform(test_data)
    test_rmse = evaluator.evaluate(test_predictions)
    
    # -------------------------------------------------------------------------
    
    # ---------------------- Put results in res dict --------------------------
    res = {
        'test_rmse': test_rmse,
        'valid_rmse_depth_5': valid_rmses[5],
        'valid_rmse_depth_7': valid_rmses[7],
        'valid_rmse_depth_9': valid_rmses[9],
        'valid_rmse_depth_12': valid_rmses[12]
    }
    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_8')
    return res
    # -------------------------------------------------------------------------

In [ ]:
res = task_8(data_io, data_dict['ml_features_train'], data_dict['ml_features_test'])
pa2.tests.test(res, 'task_8')

In [ ]:
print ("End to end time: {}".format(time.time()-begin))